Please note that Colaboratory is not stable, so it would be helpfult to save your code frequently.


--- Change Colaboratory to GPU mode


*   Runtime => Change runtime type => Hardware accelerator：Choose GPU
*   You may need to re-install torch or MPol each time you change the setting to GPU/CPU

Don't worried if Colaboratory indicates that you cannot change runtime into GPU, that's because the GPU resource of a google account has a limit, thus if you ran your code with GPU too much in a very short time, you might need to wait until you have GPU resource for your account again. 






In [1]:
# install torch and MpoL, even you might already have installed torch or MpoL on your device, you still need to install them for Colaboratory

In [ ]:
pip install torch

In [ ]:
pip install MPoL

In [ ]:
# test if installed successfully，it should return false if runtime is CPU
import torch
print(torch.cuda.is_available())

True


A sample to test the difference between runing your code on CPU and GPU



In [ ]:
# if your setting is correct, you should see a result: cuda:0, which means you successfully changed the setting to GPU
# if you are still running your code on CPU, the result would be: cpu
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [ ]:
# for one channel 
# run this code on CPU: 7s
# run this code on GPU: 6s

# for all channels: 
# run this code on CPU: 15s
# run this code on GPU: 5s
from astropy.utils.data import download_file
import numpy as np
from mpol import gridding, coordinates
fname = download_file(
    'https://zenodo.org/record/4498439/files/logo_cube.npz',
    cache=True,
    )
d = np.load(fname)
coords = coordinates.GridCoords(cell_size=0.03, npix=180)
#central_chan = 4
gridder = gridding.Gridder(
    coords=coords,
    uu=d['uu'],
    vv=d['vv'],
    weight=d['weight'],
    data_re=d['data_re'],
    data_im=d['data_im'],
)
dataset = gridder.to_pytorch_dataset()

In [ ]:
from mpol.precomposed import SimpleNet
model = SimpleNet(coords=coords, nchan=dataset.nchan)

In [ ]:
# run this code on CPU: 
# run this code on GPU: 0 s
dset = dataset.to(device)
model = model.cuda()
config = {'lr':0.5, 'lambda_sparsity':1e-4, 'lambda_TV':1e-4, 'epochs':600}
optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])

In [ ]:
# for one chennel:
# run this code on CPU: 5 s
# run this code on GPU: 1 s

# for one chennel:
# run this code on CPU: 1 min
# run this code on GPU: 2 s
from mpol import losses

# set the model to training mode
model.train()

for i in range(config['epochs']):
    # set the model to zero grad
    model.zero_grad()

    # forward pass
    vis = model.forward()

    # get skycube from our forward model
    sky_cube = model.icube.sky_cube

    # compute loss
    loss = (
        losses.nll_gridded(vis, dset)
        + config['lambda_sparsity'] * losses.sparsity(sky_cube)
        + config['lambda_TV'] * losses.TV_image(sky_cube))

    # perform a backward pass
    loss.backward()

    # update the weights
    optimizer.step()